# 03 — Decoder Comparison

Compare Union-Find, PyMatching (MWPM), and Fusion Blossom decoders on the same
syndrome shots: logical error rates and decode latency distributions.

**Requirements:** `pip install stabstream numpy matplotlib`  
**Optional decoders:**
- PyMatching: `pip install pymatching`
- Fusion Blossom: compiled Rust feature (`cargo build --features mwpm`)

Decoders that are unavailable are skipped automatically.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# --- Configuration ---
RECORDING_PATH = None  # e.g. "../data/surface_d5.qssf"
DEM_PATH = None        # e.g. "../models/surface_d5.dem"
N_SHOTS = 500          # shots to decode
WINDOW_DEPTH = 5       # syndrome rounds per decode call
N_ANCILLAS = 24        # d=5 surface code
P_PHYSICAL = 0.005

## 1. Load or generate syndrome data

In [ ]:
def load_windows_from_file(path: str, depth: int) -> list[np.ndarray]:
    """Load syndrome windows from a QSSF file."""
    from stabstream import SyndromeWindow, StabstreamStream
    windows = []
    window = SyndromeWindow(ancilla_count=N_ANCILLAS, window_depth=depth)
    with StabstreamStream(path) as stream:
        for frame in stream:
            window.push(frame)
            if window.is_full():
                windows.append(window.to_numpy_matrix().copy())
    return windows


def synthetic_windows(
    n_shots: int, depth: int, ancillas: int, p: float, seed: int = 42
) -> list[np.ndarray]:
    """Generate random (depth, ancillas) syndrome windows."""
    rng = np.random.default_rng(seed)
    return [rng.random((depth, ancillas)) < 2 * p for _ in range(n_shots)]


def synthetic_ground_truth(n_shots: int, p_l: float = 0.02, seed: int = 99) -> np.ndarray:
    """Generate random ground-truth observable flip bitmasks."""
    rng = np.random.default_rng(seed)
    return (rng.random(n_shots) < p_l).astype(np.uint64)


if RECORDING_PATH and Path(RECORDING_PATH).exists():
    windows = load_windows_from_file(RECORDING_PATH, WINDOW_DEPTH)[:N_SHOTS]
    print(f"Loaded {len(windows)} windows from {RECORDING_PATH}")
else:
    windows = synthetic_windows(N_SHOTS, WINDOW_DEPTH, N_ANCILLAS, P_PHYSICAL)
    print(f"Synthetic: {len(windows)} windows ({WINDOW_DEPTH} rounds × {N_ANCILLAS} ancillas)")

ground_truth = synthetic_ground_truth(len(windows))

## 2. Load DEM / build graph

In [ ]:
dem = None
graph = None

if DEM_PATH and Path(DEM_PATH).exists():
    try:
        from stabstream import DetectorErrorModel
        dem = DetectorErrorModel.from_file(DEM_PATH)
        print(f"DEM: {dem.detector_count} detectors, {dem.error_count} errors")
    except Exception as e:
        print(f"Could not load DEM: {e}")
else:
    print("No DEM file provided — PyMatching and FusionBlossom will use synthetic fallback.")

## 3. Benchmark each decoder

In [ ]:
def decode_with_timing(
    decode_fn,
    windows: list[np.ndarray],
    ground_truth: np.ndarray,
) -> dict:
    """Run decode_fn on each window, record latency and logical errors."""
    latencies_ns = []
    n_errors = 0
    n_shots = len(windows)

    for i, w in enumerate(windows):
        t0 = time.perf_counter_ns()
        result = decode_fn(w)
        t1 = time.perf_counter_ns()
        latencies_ns.append(t1 - t0)
        predicted = result.get("observable_flips", 0)
        if int(predicted) != int(ground_truth[i]):
            n_errors += 1

    lats = np.array(latencies_ns, dtype=float)
    p_l = n_errors / n_shots
    return {
        "latencies_ns": lats,
        "p_l": p_l,
        "n_errors": n_errors,
        "n_shots": n_shots,
        "p50_ns": float(np.percentile(lats, 50)),
        "p99_ns": float(np.percentile(lats, 99)),
    }


results = {}

# --- Null decoder (baseline: never correct) ---
def null_decode(w):
    return {"observable_flips": 0}

results["Null"] = decode_with_timing(null_decode, windows, ground_truth)
print(f"Null:  p_L={results['Null']['p_l']:.4f}  p50={results['Null']['p50_ns']:.0f} ns")

In [ ]:
# --- Union-Find decoder (pure Python simulation for comparison) ---
# In production use the Rust decoder via stabstream bindings.
# Here we simulate UF with a simple majority-vote proxy for demonstration.

def union_find_proxy(w: np.ndarray) -> dict:
    """Proxy: flag observable flip if any ancilla fires in >50% of rounds."""
    fire_rate = w.mean(axis=0)
    flipped = int(fire_rate.max() > 0.5)
    return {"observable_flips": flipped}

try:
    # Try the real Rust UF decoder if available
    from stabstream._stabstream import UnionFindDecoder  # type: ignore
    if graph is not None:
        uf_decoder = UnionFindDecoder(graph)
        def uf_decode(w):
            result = uf_decoder.decode_matrix(w)
            return {"observable_flips": result.observable_flips}
    else:
        uf_decode = union_find_proxy
except ImportError:
    uf_decode = union_find_proxy

results["Union-Find"] = decode_with_timing(uf_decode, windows, ground_truth)
r = results["Union-Find"]
print(f"UF:    p_L={r['p_l']:.4f}  p50={r['p50_ns']:.0f} ns  p99={r['p99_ns']:.0f} ns")

In [ ]:
# --- PyMatching (MWPM) ---
try:
    if dem is not None:
        from stabstream.decoders import PyMatchingDecoder
        pm = PyMatchingDecoder(dem)
        def pm_decode(w):
            return pm.decode(w)
    else:
        # Minimal pymatching example with a repetition code
        import pymatching
        import networkx as nx
        # 1-D repetition code on N_ANCILLAS detectors
        H = np.zeros((N_ANCILLAS, N_ANCILLAS + 1), dtype=np.uint8)
        for i in range(N_ANCILLAS):
            H[i, i] = 1
            H[i, i + 1] = 1
        matching = pymatching.Matching(H)
        def pm_decode(w):
            defects = w.any(axis=0).astype(np.uint8)
            correction = matching.decode(defects)
            return {"observable_flips": int(correction.sum() % 2)}

    results["PyMatching"] = decode_with_timing(pm_decode, windows, ground_truth)
    r = results["PyMatching"]
    print(f"PyMatch: p_L={r['p_l']:.4f}  p50={r['p50_ns']:.0f} ns  p99={r['p99_ns']:.0f} ns")
except ImportError:
    print("pymatching not installed — skipping (pip install pymatching)")

In [ ]:
# --- Fusion Blossom (Rust MWPM, requires --features mwpm) ---
try:
    from stabstream._stabstream import FusionBlossomDecoder  # type: ignore
    if graph is not None:
        fb = FusionBlossomDecoder(graph)
        def fb_decode(w):
            result = fb.decode_matrix(w)
            return {"observable_flips": result.observable_flips}
        results["FusionBlossom"] = decode_with_timing(fb_decode, windows, ground_truth)
        r = results["FusionBlossom"]
        print(f"FusionBlossom: p_L={r['p_l']:.4f}  p50={r['p50_ns']:.0f} ns")
    else:
        print("FusionBlossom: DEM required — skipping.")
except ImportError:
    print("FusionBlossom not available (requires --features mwpm at build time).")

## 4. Summary table

In [ ]:
print(f"{'Decoder':<16} {'p_L':>8} {'p50 (ns)':>12} {'p99 (ns)':>12} {'n_errors':>10}")
print("-" * 62)
for name, r in results.items():
    print(f"{name:<16} {r['p_l']:>8.4f} {r['p50_ns']:>12.0f} {r['p99_ns']:>12.0f} "
          f"{r['n_errors']:>10}")

## 5. Latency distributions

In [ ]:
from stabstream.plot import plot_latency_hist

# Exclude Null decoder from latency plot — it's trivially fast and not interesting
latency_data = {
    name: r["latencies_ns"]
    for name, r in results.items()
    if name != "Null"
}

if latency_data:
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_latency_hist(
        latency_data,
        ax=ax,
        title="Decode Latency Distribution",
        bins=50,
        log_x=True,
    )
    # Add 1 µs deadline line (typical superconducting syndrome cycle)
    ax.axvline(1000, color="red", linestyle="--", linewidth=1.5,
               alpha=0.7, label="1 µs deadline")
    ax.legend()
    plt.tight_layout()
    plt.savefig("latency_comparison.png", dpi=150)
    plt.show()

## 6. p_L comparison bar chart

In [ ]:
names = list(results.keys())
p_ls = [results[n]["p_l"] for n in names]
errors = [
    np.sqrt(results[n]["p_l"] * (1 - results[n]["p_l"]) / results[n]["n_shots"])
    for n in names
]

fig, ax = plt.subplots(figsize=(6, 4))
colors = ["#888888", "#4363D8", "#E6194B", "#3CB44B"]
bars = ax.bar(names, p_ls, color=colors[:len(names)], width=0.5)
ax.errorbar(range(len(names)), p_ls, yerr=errors, fmt="none",
            color="black", capsize=4, linewidth=1.5)
ax.set_ylabel("Logical error rate  p_L")
ax.set_title(f"Logical Error Rate by Decoder (p={P_PHYSICAL}, {N_SHOTS} shots)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("p_l_comparison.png", dpi=150)
plt.show()